# 02 · Build all comma2k19 processed data

Resume-safe: existing video+metadata pairs are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive/Blackbox-Detection')
REPO = Path('/content/Blackbox-Detection')
DATA_ROOT = DRIVE_ROOT / 'DATASET'
COMMA_ROOT = DATA_ROOT / 'comma2k19'
RAW_ROOT = COMMA_ROOT / 'raw'
PROCESSED_ROOT = COMMA_ROOT / 'processed' / 'v1'
MANIFEST_ROOT = DRIVE_ROOT / 'manifests' / 'stage3' / 'v1'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs' / 'stage3'
PRETRAINED_ROOT = DRIVE_ROOT / 'pretrained'

# Clone your repository if this runtime does not have it yet.
if not REPO.exists():
    raise RuntimeError('Clone Blackbox-Detection to /content/Blackbox-Detection first, then rerun this cell.')
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

for p in [PROCESSED_ROOT, MANIFEST_ROOT, OUTPUT_ROOT, PRETRAINED_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('RAW_ROOT      :', RAW_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)
# Install this repository through its existing pyproject.toml without replacing
# Colab's binary stack. Dependency versions in pyproject.toml are aligned to the
# DACON evaluation-server package list.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)],
    check=True,
)


In [ ]:
import pandas as pd
from blackbox_detection.stage3.comma2k19 import find_archives, PrepareConfig, prepare_archive
from blackbox_detection.stage3.manifest import build_segment_manifest

archives = find_archives(RAW_ROOT)
cfg = PrepareConfig(processed_root=PROCESSED_ROOT, overwrite=False)
all_reports = []
for i, archive in enumerate(archives, 1):
    print(f'[{i}/{len(archives)}] {archive.name}')
    rep = prepare_archive(archive, cfg)
    all_reports.append(rep)
    bad = rep[rep.get('error', pd.Series(index=rep.index, dtype=object)).notna()] if 'error' in rep else pd.DataFrame()
    if len(bad):
        display(bad)

report = pd.concat(all_reports, ignore_index=True)
report.to_csv(PROCESSED_ROOT / 'prepare_report.csv', index=False)
manifest = build_segment_manifest(PROCESSED_ROOT)
manifest.to_csv(PROCESSED_ROOT / 'manifest.csv', index=False)
print('segments:', len(manifest), 'frames:', manifest.num_frames.sum())
display(manifest.head())

In [ ]:
import shutil
usage = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free: {usage.free/2**30:.1f} GiB')
print('Processed video GiB:', sum(p.stat().st_size for p in (PROCESSED_ROOT/'videos').rglob('*.mp4'))/2**30)
print('Metadata GiB       :', sum(p.stat().st_size for p in (PROCESSED_ROOT/'metadata').rglob('*.npz'))/2**30)